In [ ]:
pre_forward()  # 等待 chunk1 的输入到达
    
# chunk1 处理 micro-batch 0
forward_output_tensor = forward_step_helper(3, checkpoint)

post_forward(forward_output_tensor)  # 发送（如果是None就不发）

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│              Steady 1F1B 阶段                                │
├─────────────────────────────────────────────────────────────┤
│  前向: pre_forward → forward_step → post_forward            │
│       ↑本代码所在位置  ↑计算      ↑发送输出+预接收              │
├─────────────────────────────────────────────────────────────┤
│  反向: pre_backward → backward_step → post_backward         │
│       ↑等待梯度      ↑计算      ↑发送梯度+预接收              │
└─────────────────────────────────────────────────────────────┘

# pre_forward
- 这段代码是 VPP 稳态阶段（1F1B）前向传播前的同步预处理，
- 核心目的是<font color='red'>确保前向计算所需的输入张量已从上游 stage 接收完毕，并释放上一个迭代的输出张量以节省内存</font>

## get_model_chunk_id中forward=True的含义
- 前向传播时的 chunk ID 查找方向

In [ ]:
调度表（正向）:
v_id | 0  1  2  3  4  5  6  7  8  9
mb   | 0  1  2  0  1  2  3  4  3  4
ch   | 0  0  0  1  1  1  0  0  1  1

示例: forward_k=3
  get_model_chunk_id(3, forward=True) → model_chunk_id=1

| 列      | 含义                | 取值范围  |
| ------ | ----------------- | ----- |
| `v_id` | 虚拟微批次 ID（执行顺序）    | 0 ~ 9 |
| `mb`   | 真实微批次 ID（数据批次）    | 0 ~ 4 |
| `ch`   | 模型 chunk ID（虚拟阶段） | 0, 1  |


从表结构反推配置：

| 参数                                       | 值     | 推导                  |
| ---------------------------------------- | ----- | ------------------- |
| `num_microbatches` (M)                   | **5** | mb 最大值为 4           |
| `len(model)` (VPP)                       | **2** | ch 取值为 0, 1         |
| `microbatch_group_size_per_vp_stage` (G) | **3** | 每组 3 个连续 v\_id 同 ch |
| `pipeline_parallel_size` (PP)            | **2** | 见下文分析               |

### 执行顺序解读
- 按 v_id 顺序执行（从左到右）：

In [ ]:
v_id=0:  mb=0, ch=0  →  用 micro-batch 0 的数据，跑 chunk 0（虚拟阶段0）
v_id=1:  mb=1, ch=0  →  用 micro-batch 1 的数据，跑 chunk 0
v_id=2:  mb=2, ch=0  →  用 micro-batch 2 的数据，跑 chunk 0

v_id=3:  mb=0, ch=1  →  用 micro-batch 0 的数据，跑 chunk 1（虚拟阶段1）
v_id=4:  mb=1, ch=1  →  用 micro-batch 1 的数据，跑 chunk 1
v_id=5:  mb=2, ch=1  →  用 micro-batch 2 的数据，跑 chunk 1

v_id=6:  mb=3, ch=0  →  用 micro-batch 3 的数据，跑 chunk 0
v_id=7:  mb=4, ch=0  →  用 micro-batch 4 的数据，跑 chunk 0

v_id=8:  mb=3, ch=1  →  用 micro-batch 3 的数据，跑 chunk 1
v_id=9:  mb=4, ch=1  →  用 micro-batch 4 的数据，跑 chunk 1

### 分组模式
- 关键规律：同一 mb 的两个 ch 不连续执行，中间隔了 G=3 个 v_id 

In [ ]:
Group 0 (v_id 0-2):  ch=0, mb=0,1,2  ← 3个micro-batch跑chunk0
Group 1 (v_id 3-5):  ch=1, mb=0,1,2  ← 同样的3个micro-batch跑chunk1
    
Group 2 (v_id 6-7):  ch=0, mb=3,4    ← 剩余2个micro-batch跑chunk0
Group 3 (v_id 8-9):  ch=1, mb=3,4    ← 同样的2个micro-batch跑chunk1

### 为什么这样设计？
#### 目标：最大化流水线填充

In [ ]:
传统PP（无VPP）的问题：
GPU 0: [F0]        [F1]        [F2]        [F3]        [F4]
GPU 1:      [F0]        [F1]        [F2]        [F3]        [F4]
           ↑ 每个时刻只有1个GPU在忙（假设单chunk）

VPP交错调度：
GPU 0 (ch0+ch1): [F0_ch0][F1_ch0][F2_ch0] [F0_ch1][F1_ch1][F2_ch1] ...
GPU 1 (ch0+ch1):      [F0_ch0][F1_ch0] ...

#### 同一 mb 的两个 chunk 分离的原因

In [ ]:
mb=0 的完整前向传播：

  chunk0 (v_id=0) → 输出传给 chunk1 (v_id=3)
  
  如果 v_id=0 之后立即 v_id=3（同一mb的chunk1）：
    → chunk0 的输出还没传给下一个stage（需要P2P通信）
    → 或者下一个stage还没算完上一个mb的chunk0
    
  间隔 G=3 个 v_id：
    → 给通信和下游stage留出时间
    → 形成"流水线填充"效果

### 为什么首阶段不需要等待接收输入？

In [ ]:
PP 拓扑:
GPU 0 (Stage 0): chunk0 (VPP首), chunk2 (VPP尾)  ← PP 首阶段
GPU 1 (Stage 1): chunk1 (VPP中), chunk3 (VPP末)

前向传播方向: 从首阶段往末阶段传
  Stage 0 (GPU 0): 
    - chunk0 的输入来自数据加载器（外部输入），不是 P2P 接收
    - 不需要等待 recv_prev！
    
  Stage 1 (GPU 1):
    - chunk1 的输入来自 GPU 0 的 P2P 发送
    - 需要等待 recv_prev 完成！

### recv_prev_wait_handles.pop(0) ,为什么是 pop(0)（FIFO）？

In [ ]:
时间线:
t0: post_forward() 发起预接收 micro-batch 0 的输入 → handle_A
t1: post_forward() 发起预接收 micro-batch 1 的输入 → handle_B
t2: pre_forward() 需要 micro-batch 0 的输入 → pop(0) 得 handle_A ✓
t3: pre_forward() 需要 micro-batch 1 的输入 → pop(0) 得 handle_B ✓

如果 pop()（LIFO）:
t2: pop() 得 handle_B → 但需要的是 micro-batch 0！✗ 顺序错乱

In [ ]:

            recv_prev_wait_handles = []
        
            '''
            pp_pre_forward：前向预处理
            '''
            def pp_pre_forward(vp_stage=None):
            
                '''
                确定当前虚拟阶段:
                默认调用	get_model_chunk_id(forward_k, forward=True)
                显式传入	外层直接指定（如 MoE 特殊调度）
                '''
                if vp_stage is None:
                    vp_stage = get_model_chunk_id(forward_k, forward=True)
                
                '''1. 等待接收前向输入（同步）'''
                '''
                _is_vp_first_stage(vp_stage)	当前 chunk 是 VPP 的第一个虚拟阶段
                is_pp_first_stage(pp_group)	当前 GPU 是 PP 的第一个物理阶段
                and	同时满足 = 真正的输入端
                
                '''
                if not (_is_vp_first_stage(vp_stage=vp_stage) and is_pp_first_stage(pp_group)):
                    '''
                    等待接收句柄（重叠通信模式）
                    
                    特点：Warmup 阶段就开始预取输入，Flush 机制确保顺序
                    '''
                    if config.overlap_p2p_comm_warmup_flush:
                        
                        '''
                        保护场景：如果列表为空，说明前置的 post_forward 没有正确发起预接收
                        '''
                        assert recv_prev_wait_handles, (
                            f'哪个 GPU 出错:pp rank {pipeline_parallel_rank}, 前向第几次迭代:fwd iteration {forward_k}, '
                            'should have registered recv handle'
                        )
                        '''
                        
                        '''
                        recv_prev_wait_handle = recv_prev_wait_handles.pop(0) # FIFO 队列
                        recv_prev_wait_handle.wait() # 阻塞等待输入到达
                    else:
                        '''
                        等待接收句柄（常规重叠模式）
                        
                        
                        '''
                        if recv_prev_wait_handles is not None and recv_prev_wait_handles:
                            recv_prev_wait_handle = recv_prev_wait_handles.pop(0)
                            recv_prev_wait_handle.wait()

                ''' 2. 释放上一个输出张量（省内存）'''
                '''
                
                注意：这里释放的是上一次迭代的 output_tensor，不是当前的！
                
                
                为什么在这里释放？
                
                1F1B 时序:
                  迭代 k:
                    pre_forward()      ← 等待本次输入
                    forward_step()     ← 计算本次输出 → 保存到 output_tensor
                    post_forward()     ← 发送 output_tensor，预接收下次输入

                  迭代 k+1:
                    pre_forward()      ← 等待下次输入（已预接收）
                    deallocate_output_tensor(output_tensor)  ← 释放上次迭代的输出！
                    forward_step()     ← 计算新输出

                  关键点: output_tensor 在 post_forward 发送后、pre_forward 释放前
                          这段时间内，output_tensor 已完成发送，可以安全释放
                '''
                
                '''
                config.deallocate_pipeline_outputs 控制:
                
                 - True	立即释放，节省内存
                 - False	保留，可能用于调试或其他用途
                '''
                deallocate_output_tensor(output_tensor, config.deallocate_pipeline_outputs)


# post_forward
- 这段代码是 VPP 稳态阶段（1F1B）前向传播后的异步通信处理，
- 核心目的是<font color='red'>发送当前输出给下游 stage，同时预接收下一个输入</font>——实现计算与通信重叠。

我来逐层拆解：

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│              Steady 1F1B 阶段                                │
├─────────────────────────────────────────────────────────────┤
│  前向: pre_forward → forward_step → post_forward            │
│       ↑等待输入       ↑计算           ↑本代码所在位置              │
├─────────────────────────────────────────────────────────────┤
│  反向: pre_backward → backward_step → post_backward         │
│       ↑等待梯度             ↑计算      ↑发送梯度+预接收              │
└─────────────────────────────────────────────────────────────┘

## 闭包变量声明:

| 变量                       | 来源   | 用途                        |
| ------------------------ | ---- | ------------------------- |
| `send_next_wait_handle`  | 外层函数 | 缓存的"发送给下游"异步句柄            |
| `fwd_recv_buffer`        | 外层函数 | 前向接收缓冲区（循环复用）             |
| `fwd_wait_handles`       | 外层函数 | 当前 P2P 操作的句柄字典            |
| `recv_prev_wait_handles` | 外层函数 | 供 `pre_forward` 消费的接收句柄队列 |


## 完整时序示例
### 场景：PP=2, VPP=2, GPU 0 (Stage 0)

In [ ]:
Stage 0 (GPU 0): chunk0 (VPP首), chunk2 (VPP尾)
Stage 1 (GPU 1): chunk1 (VPP中), chunk3 (VPP末)

### Steady 迭代 0 (k=0, forward_k=3)

In [ ]:
前置状态:
  send_next_wait_handle = handle_from_warmup_last  ← 上次的发送句柄
  recv_prev_wait_handles = []  ← pre_forward 已消费完

1. forward_step_helper() 计算完成
   output_tensor = [激活值]  ← chunk2 的输出

2. 进入 pp_post_forward(output_tensor):
   
   a. vp_stage = get_model_chunk_id(3, forward=True) → 1 (chunk2)
   
   b. _is_vp_last_stage(1)? chunk2 是 VPP尾 → True
      is_pp_last_stage(pp_group)? GPU 0 是 Stage 0 → False
      True and False = False → 不是真正的末阶段！
      output_tensor 保持不变（需要发送给 GPU 1）
   
   c. recv_prev, next_chunk = recv_tensor_from_previous_stage(3, True)
      → recv_prev = False? 查调度表...
      实际上 GPU 0 的 chunk2 输入来自哪里？
      → 来自 GPU 0 自己的 chunk0（同一 GPU 内，非 P2P）
      → recv_prev = False（不需要 P2P 接收）
   
   d. k=0, num_microbatches_remaining-1 = ?
      假设 remaining=2, 则 0 != 1 → recv_prev 保持
   
   e. 发起 P2P:
      fwd_recv_buffer[3 % 1], fwd_wait_handles = send_forward_recv_forward(
          output_tensor,          # 发送给 GPU 1
          recv_prev=False,        # 不需要接收
          tensor_shape, overlap_p2p_comm=True
      )
      → 只发送，不接收
      → fwd_wait_handles = {"send_next": handle_send}
   
   f. 等待上次发送:
      send_next_wait_handle.wait()  ← 等 warmup 最后一次发送完成
   
   g. 提取句柄:
      send_next_wait_handle = fwd_wait_handles.pop("send_next")  ← 本次发送句柄
      "recv_prev" not in handles → 不 append
   
   h. recv_prev=False → 不缓存输入

3. 返回 output_tensor（可能被修改，但这里没变）

### 与 pp_pre_forward 的配合

In [ ]:
迭代 k:
  pre_forward():
    wait(recv_prev_wait_handles[0])  ← 等 post_forward_{k-1} 预接收的输入
    deallocate(output_tensor_{k-1})
  
  forward_step():
    compute → output_tensor_k
  
  post_forward():
    send(output_tensor_k)            ← 发起异步发送
    recv(input_{k+1})               ← 预接收下次输入
    wait(send_{k-1})                ← 等上次发送完成
    append(recv_handle_{k+1})       ← 缓存给 pre_forward_{k+1}

迭代 k+1:
  pre_forward():
    wait(recv_prev_wait_handles[0])  ← 等 post_forward_k 预接收的输入
    ...

In [ ]:

            '''前向后处理'''
            # Async forward send / receive
            def pp_post_forward(output_tensor, vp_stage=None):
                '''
                闭包变量声明:
                '''
                nonlocal send_next_wait_handle
                nonlocal fwd_recv_buffer
                nonlocal fwd_wait_handles
                nonlocal recv_prev_wait_handles
                '''
                确定当前虚拟阶段:
                '''
                if vp_stage is None:
                    vp_stage = get_model_chunk_id(forward_k, forward=True)
                    
                '''1. 末阶段截断'''
                '''
                - _is_vp_last_stage(vp_stage)	当前 chunk 是 VPP 最后一个虚拟阶段
                - is_pp_last_stage(pp_group)	当前 GPU 是 PP 最后一个物理阶段
                - and	同时满足 = 真正的输出端，无下游可发送


                '''
                # Last virtual stage no activation tensor to send.
                if _is_vp_last_stage(vp_stage=vp_stage) and is_pp_last_stage(pp_group):
                    '''
                    PP 拓扑:
                     GPU 0 (Stage 0): chunk0, chunk2
                     GPU 1 (Stage 1): chunk1, chunk3  ← PP 末阶段

                     Stage 1 的 chunk3 前向输出:
                       → 传给 LM Head 计算 loss
                       → 不需要 P2P 发送给下一个 stage（因为没有下一个 stage）
  
                       output_tensor = None → P2P 发送函数看到 None，跳过发送
                    
                    '''
                    output_tensor = None

                '''
                判断是否需要接收下一个输入:
                
                - recv_prev	是否需要从上游 stage 接收输入
                - next_forward_model_chunk_id	下次前向的 chunk ID
                '''
                recv_prev, next_forward_model_chunk_id = recv_tensor_from_previous_stage(
                    forward_k, forward=True
                )

                # If last iteration, don't receive; we already received one extra
                # before the start of the for loop.
                '''
                最后一次迭代特殊处理:
                  原因：Steady 循环开始前已预接收了一个额外的输入（Warmup 阶段的末尾），最后一次迭代不需要再接收。
                '''
                
                '''
                预接收机制:
                Warmup 结束前:
                  post_forward() 已发起 recv for steady_0 的输入
  
                Steady 迭代 0:
                  pre_forward() 使用预接收的输入
                  post_forward() 发起 recv for steady_1 的输入
  
                ...

                Steady 迭代 last:
                  pre_forward() 使用上次预接收的输入
                  post_forward() 不需要再 recv（无下次迭代）
                  
                '''
                if k == (num_microbatches_remaining - 1):
                    recv_prev = False

                # Send activation tensor to the next stage and receive activation tensor from the
                # previous stage
                
                '''2. 发起异步 P2P（发送输出 + 预接收下一个输入）'''
                
                '''
                send_forward_recv_forward 行为:
                
                输入参数:
                  output_tensor = [batch, seq, hidden] 或 None
                  recv_prev = True/False
                  tensor_shape = [seq_len, micro_batch_size, hidden_size]

                内部逻辑:
                  if output_tensor is not None:
                      send_handle = isend(output_tensor, dst=next_stage)  # 异步发送
  
                  if recv_prev:
                      recv_buffer = empty(tensor_shape)                   # 预分配接收缓冲
                      recv_handle = irecv(recv_buffer, src=prev_stage)    # 异步接收
                      return recv_buffer, {"send_next": send_handle, "recv_prev": recv_handle}
                  else:
                      return None, {"send_next": send_handle}
                '''
                '''
                缓冲区循环复用: fwd_recv_buffer[forward_k % fwd_recv_buffer_size]
                
                fwd_recv_buffer_size = 2（假设）

                迭代 0: forward_k=3 → 3 % 2 = 1 → 使用 buffer[1]
                迭代 1: forward_k=4 → 4 % 2 = 0 → 使用 buffer[0]
                迭代 2: forward_k=5 → 5 % 2 = 1 → 复用 buffer[1]（迭代0的数据已消费）

                循环复用，避免无限分配内存
                '''
                fwd_recv_buffer[forward_k % fwd_recv_buffer_size], fwd_wait_handles = (
                    p2p_communicator.send_forward_recv_forward(
                        output_tensor,         # 要发送的输出（None=不发送）
                        recv_prev=recv_prev,   # 是否接收输入
                        tensor_shape=tensor_shape,
                        overlap_p2p_comm=True, # 启用异步重叠
                    )
                )
                
                '''3. 等待上一次发送完成（避免积压）'''
                
                '''
                为什么需要等待？
                
                发送队列堆积问题:
                t0: post_forward() 发起 send_A → send_next_wait_handle = handle_A
                t1: post_forward() 发起 send_B → 如果 handle_A 还没完成？
                    → 发送队列堆积 → 内存爆炸或 NCCL 错误

                解决方案: 每次发起新发送前，等待上次发送完成
                          → 保证最多只有一个未完成的发送
                          
                时序:
                迭代 k:
                  post_forward():
                    wait(send_next_wait_handle_{k-1})  ← 等上次发送完成
                    发起 send_k → send_next_wait_handle = handle_k

                迭代 k+1:
                  post_forward():
                    wait(send_next_wait_handle_k)      ← 等本次发送完成
                    发起 send_{k+1}
                '''
                '''4. 缓存接收句柄（供下次 pre_forward 使用）'''
                # 提取发送句柄，留给下次迭代等待
                if send_next_wait_handle is not None:
                    send_next_wait_handle.wait()
                    
                '''
                fwd_wait_handles = {
                    "send_next": handle_send,    # 发送给下游的句柄
                    "recv_prev": handle_recv     # 从上游接收的句柄
                }

                .pop("send_next") → 给 send_next_wait_handle（本函数下次迭代用）
                .pop("recv_prev") → append 到 recv_prev_wait_handles（pre_forward 用）
                '''
                if fwd_wait_handles is not None:
                    send_next_wait_handle = (
                        fwd_wait_handles.pop("send_next")
                        if "send_next" in fwd_wait_handles
                        else None
                    )
                    
                    
                    # 提取接收句柄，留给下次 pre_forward 等待
                    if "recv_prev" in fwd_wait_handles:
                        # 提取发送句柄，留给下次迭代等待
                        recv_prev_wait_handles.append(fwd_wait_handles.pop("recv_prev"))
                # assert fwd_wait_handles is not None

                # Put input_tensor and output_tensor_grad in data structures in the
                # right location.
                '''
                缓存接收到的输入
                '''
                
                '''
                迭代 0: forward_k=3
                  使用 buffer[3 % 2] = buffer[1]
                  清空 buffer[(3+1) % 2] = buffer[0]  ← 迭代1要用的位置

                迭代 1: forward_k=4
                  使用 buffer[4 % 2] = buffer[0]（刚清空）
                  清空 buffer[(4+1) % 2] = buffer[1]  ← 迭代2要用的位置

                防止旧数据干扰
                
                '''
                if recv_prev:
                    input_tensors[next_forward_model_chunk_id].append(
                        fwd_recv_buffer[forward_k % fwd_recv_buffer_size]
                    )
                    '''
                    清空下次要用的缓冲槽
                    '''
                    fwd_recv_buffer[(forward_k + 1) % fwd_recv_buffer_size] = None

                return output_tensor


# forward_step_helper 执行实际的前向计算
- 这段代码是 Megatron-LM 中 VPP（虚拟流水线并行）的前向传播核心辅助函数，
- <font color='red'>负责将虚拟微批次映射到具体的模型 chunk 和数据，执行实际的前向计算。</font>

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│              forward_step_helper                             │
│              （虚拟微批次 → 真实计算）                         │
├─────────────────────────────────────────────────────────────┤
│  1. 查表：v_id → model_chunk_id + microbatch_id             │
│  2. 预处理：准备输入张量                                     │
│  3. 计算：调用 forward_step 执行真实前向                     │
│  4. 后处理：保存输出和 token 计数                            │
│  5. 返回：输出张量                                           │
└─────────────────────────────────────────────────────────────┘

## 关键参数详解

| 参数                                  | 作用                                                        |
| ----------------------------------- | --------------------------------------------------------- |
| `forward_step_func`                 | 用户传入的函数，定义如何调用模型（如 `model(input_ids, position_ids, ...)`） |
| `data_iterator[model_chunk_id]`     | VPP 每个 chunk 有独立的数据迭代器                                    |
| `model[model_chunk_id]`             | 当前虚拟阶段的模型分片（如 `GPTModel(layers=0-3)`）                     |
| `checkpoint_activations_microbatch` | 是否对此微批次启用激活检查点（省内存、多计算）                                   |
| `is_first_microbatch`               | 影响某些层的初始化（如 dropout、batch norm）                           |
| `current_microbatch`                | 用于 loss scaling 和日志记录                                     |
| `vp_stage`                          | 虚拟阶段 ID，用于阶段间通信判断                                         |
| `is_last_stage`                     | 最终阶段需要计算 loss，其他阶段只返回激活值                                  |
### is_last_stage 计算

- _is_vp_last_stage(model_chunk_id)：当前 chunk 是 VPP 最后一个虚拟阶段
- is_pp_last_stage(pp_group)：当前 GPU 是 PP 最后一个物理阶段
- and：同时满足 = 真正的输出端，需要计算 loss

## 完整数据流示例
### 场景：PP=2, VPP=2, GPU 0 执行 v_id=3

In [ ]:
配置:
  model = [chunk0 (layers 0-3), chunk1 (layers 4-7)]
  data_iterator = [iter_for_chunk0, iter_for_chunk1]

执行 forward_step_helper(3, checkpoint_activations_microbatch=False):

Step 1: 查表
  model_chunk_id = get_model_chunk_id(3, forward=True) → 1
  microbatch_id = get_microbatch_id_in_model_chunk(3, forward=True) → 0
  
  含义：用数据0，跑chunk1

Step 2: 预处理
  input_tensor = forward_step_helper_preprocess(3, 1, 0)
  
  GPU 0 是 PP 首阶段，但 chunk1 不是 VPP 首阶段
  → 从 input_tensors[1].pop(0) 取输入
  → 这是之前 pre_forward 等待接收的来自 chunk0 的输出

Step 3: 计算
  output_tensor, num_tokens = forward_step(
      forward_step_func=gpt_forward,
      data_iterator=data_iterator[1],      # chunk1 的数据迭代器
      model=model[1],                       # chunk1 的模型 (layers 4-7)
      input_tensor=input_tensor,            # 来自 chunk0 的激活值
      checkpoint_activations_microbatch=False,  # 不检查点
      is_first_microbatch=False,           # chunk1 的首微批次是 v_id=3
      current_microbatch=0,                # 真实微批次0
      vp_stage=1,                          # 虚拟阶段1
      is_last_stage=False,                 # GPU 0 不是 PP 末阶段
  )
  
  内部执行:
    hidden_states = model[1](input_tensor)  # layers 4-7 前向
    output_tensor = hidden_states           # 不是末阶段，不计算 loss

Step 4: 后处理
  forward_step_helper_postprocess(1, output_tensor, num_tokens)
  
  output_tensors[1].append(output_tensor)  # 保存供反向用
  total_num_tokens += num_tokens           # 累加 token 数

Step 5: 返回
  return output_tensor  # 传给 post_forward 做 P2P 发送

### 与外层调用的关系

In [ ]:
外层 Steady 循环:
  for k in range(num_microbatches_remaining):
      forward_k = k + num_warmup_microbatches  # 如 3
      
      # 1. 前向预处理（等待输入）
      pp_pre_forward()
      
      # 2. 执行前向计算（本函数）
      output_tensor = forward_step_helper(
          forward_k,  # virtual_microbatch_id
          checkpoint_activations_microbatch
      )
      
      # 3. 前向后处理（发送输出）
      output_tensor = pp_post_forward(output_tensor)

In [ ]:

    def forward_step_helper(virtual_microbatch_id, checkpoint_activations_microbatch):
        """Helper method to run forward step with model split into chunks"""
        '''
        查表获取模型 Chunk ID:
        - 作用：将虚拟执行序号映射到实际的模型分片
        
        
        调度表示例:
        v_id | 0  1  2  3  4  5  6  7  8  9
        mb   | 0  1  2  0  1  2  3  4  3  4
        ch   | 0  0  0  1  1  1  0  0  1  1

        调用示例:
          get_model_chunk_id(3, forward=True) → 1
          get_model_chunk_id(7, forward=True) → 0
          
        forward=True：使用正向调度表（反向时 forward=False，chunk 顺序颠倒）
        '''
        model_chunk_id = get_model_chunk_id(virtual_microbatch_id, forward=True)
        '''
        查表获取真实微批次 ID
        - 作用：将虚拟执行序号映射到实际的数据批次
        
        v_id=3:
          model_chunk_id = 1（chunk1）
          microbatch_id = 0（mb0）
  
        含义：第3步执行的是"用数据0跑chunk1"
        
        '''
        microbatch_id = get_microbatch_id_in_model_chunk(virtual_microbatch_id, forward=True)

        '''
        预处理输入张量:
        '''
        input_tensor = forward_step_helper_preprocess(
            virtual_microbatch_id, model_chunk_id, microbatch_id
        )

        '''
        执行前向计算:
        
        
        '''
        output_tensor, num_tokens = forward_step(
            forward_step_func,            # 用户定义的前向函数（如 GPT 模型调用）
            data_iterator[model_chunk_id],# 当前 chunk 的数据迭代器
            model[model_chunk_id],        # 当前 chunk 的模型（nn.Module）
            num_microbatches,             # 总微批次数
            input_tensor,                 # 预处理后的输入
            forward_data_store,           # 存储中间结果
            config,                       # 全局配置
            cp_group_size=cp_size,        # 上下文并行大小
            collect_non_loss_data=collect_non_loss_data,  # 是否收集非损失数据
            checkpoint_activations_microbatch=checkpoint_activations_microbatch,# 检查点
            is_first_microbatch=check_first_val_step( # 是否是该 chunk 的首个微批次
                first_val_step,
                forward_only,
                is_first_microbatch_for_model_chunk(virtual_microbatch_id),
            ),
            current_microbatch=microbatch_id, # 当前真实微批次 ID
            vp_stage=model_chunk_id,          # 当前虚拟阶段
            is_last_stage=_is_vp_last_stage(vp_stage=model_chunk_id) and is_pp_last_stage(pp_group),# 是否是最终阶段
        )

        '''
        后处理:
        
        | 操作                                        | 说明                                   |
        | --------------------------------------     | ------------------------------------ |
        | 保存输出到 `output_tensors[model_chunk_id]` | 供后续反向传播使用                            |
        | 累加 token 数到 `total_num_tokens`          | 全局损失平均                               |
        | 记录调试信息                                 | 如激活值统计、内存使用                          |
        | 触发嵌入层权重梯度计算                        | 如果配置 `defer_embedding_wgrad_compute` |

        '''
        forward_step_helper_postprocess(model_chunk_id, output_tensor, num_tokens)

        return output_tensor


## forward_step_helper_preprocess 预处理输入张量
- 这段代码是 Megatron-LM 中 VPP（虚拟流水线并行）前向传播前的预处理函数，
- <font color='red'>负责参数同步调度、首阶段输入处理和输入张量索引。</font>
- 是前向传播前的"三重准备站"：
  - 第一，通过 v_id + rank 的提前偏移策略错峰预取下一个 chunk 的参数，让 All-Gather 与当前计算重叠；
  - 第二，为 PP 首阶段注入 None 占位符统一输入接口；
  - 第三，通过 num_released_microbatches 计算偏移量，在非深度优先的 VPP 交错调度中正确索引缓冲的输入张量——确保"该用哪份数据就用哪份数据"。

### 预处理内容：

| 操作                                       | 说明                       |
| ---------------------------------------- | ------------------------ |
| 从 `input_tensors[model_chunk_id]` 取缓存的输入 | 之前 `pre_forward` 等待接收的输入 |
| 或从 `data_iterator` 取原始输入                 | PP 首阶段、VPP 首 chunk       |
| 形状调整                                     | CP/SP 切分后的张量拼接           |
| 设备转移                                     | 确保在正确的 GPU 上             |


In [ ]:
┌─────────────────────────────────────────────────────────────┐
│         forward_step_helper_preprocess                       │
│              （前向传播预处理）                               │
├─────────────────────────────────────────────────────────────┤
│  1. 参数同步（为下一个 model chunk 预取参数）                 │
│     - 计算同步的虚拟微批次 ID（考虑 PP rank 偏移）            │
│     - 判断是否是该 chunk 的首个微批次                         │
│     - 执行参数同步（如 ZeRO-3 的 All-Gather）                │
│  2. 首阶段输入处理（PP 首阶段 + VPP 首 chunk）                │
│     - 处理外部数据输入（非 P2P 接收）                         │
│  3. 输入张量索引（支持深度优先/非深度优先调度）               │
│     - 计算偏移量，正确索引缓冲的输入张量                      │
└─────────────────────────────────────────────────────────────┘

### 参数同步 vs 梯度同步的对比

|          | 参数同步（param\_sync） | 梯度同步（grad\_sync）          |
| -------- | ----------------- | ------------------------- |
| **触发时机** | 前向传播前             | 反向传播后                     |
| **目的**   | 获取完整参数用于计算        | 聚合梯度用于更新                  |
| **方向**   | All-Gather（分片→完整） | Reduce-Scatter/All-Reduce |
| **典型场景** | ZeRO-3            | DDP/ZeRO                  |
| **错峰策略** | `v_id + rank`（提前） | `v_id - rank`（延后）         |


In [ ]:
配置:
  model = [chunk0, chunk1]
  input_tensors = [[], []]
  output_tensors = [[], []]
  config.param_sync_func = [sync_0, sync_1]  # 自定义同步

执行 forward_step_helper_preprocess(v_id=3, chunk_id=1, mb_id=0):

Step 1: 参数同步
  config.param_sync_func is not None → True
  
  param_sync_v_id = 3 + 0 = 3
  
  3 < total_num_microbatches(=10)? → True
  
  is_first_microbatch_for_model_chunk(3)?
    正向调度表: v_id=3 → ch=1, mb=0
    chunk 1 的首 mb 是 mb0（v_id=3）
    → True！
  
  param_sync_chunk_id = get_model_chunk_id(3, forward=True) + 1 = 1 + 1 = 2
  
  1 < 2 < 2? → False！（num_model_chunks=2，2 不小于 2）
  → 不执行同步
  
  （如果 num_model_chunks=4，则会同步 chunk 2）

Step 2: 首阶段检查
  _is_vp_first_stage(1)? chunk1 是 VPP尾 → False
  → 跳过 append(None)

Step 3: 输入索引
  假设 input_tensors[1] = [mb0_in_from_chunk0, mb1_in_from_chunk0, ...]
  
  offset = num_released_microbatches(3, 1)
  
  假设之前没有 mb 完成反向（还在 Warmup/Steady 早期）
  → offset = 0
  
  input_tensor = input_tensors[1][0 - 0] = input_tensors[1][0]
               = mb0_in_from_chunk0（来自 chunk0 的输出）
  
  return input_tensor

In [ ]:
    def forward_step_helper_preprocess(virtual_microbatch_id, model_chunk_id, microbatch_id):
        """Preprocess for forward_step_helper"""
        # launch param synchronization for next model chunk
        # Note: Asynchronous communication tends to slow down compute.
        # To reduce idling from mismatched microbatch times, we launch
        # asynchronous communication at the same time across the
        # pipeline-parallel group.
        '''
        参数同步调度
        条件：配置了自定义参数同步函数（如 DeepSpeed ZeRO-3 的 param_sync_func）
        '''
        if config.param_sync_func is not None:
            '''
            计算同步的虚拟微批次 ID
            
            为什么是 + pipeline_parallel_rank？（与梯度同步的 - 对比）
            
              梯度同步（postprocess）:
                grad_sync_v_id = v_id - rank  ← 延后，等慢的 rank
  
              参数同步（preprocess）:
                param_sync_v_id = v_id + rank  ← 提前，预取下一个 chunk 的参数

              原因：
                - 参数同步需要在前向计算前完成（提前准备）
                - 梯度同步需要在反向计算后触发（延后聚合）
            '''
            param_sync_virtual_microbatch_id = virtual_microbatch_id + pipeline_parallel_rank
            '''
            检查边界和首微批次条件:
            
            - < total_num_microbatches	偏移后的 v_id 不超过总虚拟微批次数
            - is_first_microbatch_for_model_chunk(...)	是该 chunk 的第一个 micro-batch
            
            为什么只在首微批次同步参数？
            
              ZeRO-3 参数分片：
                - 每个 GPU 只保存部分参数
                - 前向计算前需要 All-Gather 获取完整参数
  
                同一 chunk 的所有 micro-batch 共享同一套参数
                → 只需在第一个 mb 前同步一次
                → 后续 mb 直接使用已同步的参数
            '''
            if (
                param_sync_virtual_microbatch_id < total_num_microbatches
                and is_first_microbatch_for_model_chunk(param_sync_virtual_microbatch_id)
            ):
                '''
                计算同步的 chunk ID
                
                为什么是 + 1？
                
                目标：为"下一个" model chunk 预取参数

                   当前在 chunk 0 的前向：
                     → 预取 chunk 1 的参数（因为 chunk 1 即将使用）
  
                   当前在 chunk 1 的前向：
                     → 预取 chunk 2 的参数（如果存在）
  
                   "+1" 表示预取下一个相邻 chunk                   
                '''
                param_sync_chunk_id = (
                    get_model_chunk_id(param_sync_virtual_microbatch_id, forward=True) + 1
                )
                '''
                边界检查
                | 条件                   | 说明                               |
                | -------------------- | -------------------------------- |
                | `> 1`                | 不预取 chunk 0 和 chunk 1（可能已同步或正在用） |
                | `< num_model_chunks` | 不超过总 chunk 数                     |

                '''
                if 1 < param_sync_chunk_id < num_model_chunks:
                    '''
                    执行参数同步:
                    
                    典型实现（ZeRO-3）：

                      def param_sync_func(parameters):
                          # All-Gather 分片参数
                          for param in parameters:
                              if param.ds_tensor is not None:  # 有分片
                                  param.data = gather_from_all_dp_ranks(param.ds_tensor)
                    '''
                    config.param_sync_func[param_sync_chunk_id](
                        model[param_sync_chunk_id].parameters()
                    )

        # forward step
        '''
        首阶段输入处理
        
        _is_vp_first_stage(model_chunk_id)	当前 chunk 是 VPP 首阶段
        is_pp_first_stage(pp_group)	当前 GPU 是 PP 首阶段
        and	同时满足 = 真正的输入端
        '''
        if _is_vp_first_stage(vp_stage=model_chunk_id) and is_pp_first_stage(pp_group):
            '''
            条件判断的巧妙之处：
               初始状态: 只在初始为空的状态下才append
                 input_tensors[0] = []
                 output_tensors[0] = []
                 len([]) == len([]) → True → append(None)
  
                 input_tensors[0] = [None]

               第一次前向后:
                 output_tensors[0] = [tensor_mb0]
                 len([None]) == len([tensor_mb0]) → False → 不 append
  
               第二次前向:
                 input_tensors[0] = [None]  （如果还没消费）
                 output_tensors[0] = [tensor_mb0, tensor_mb1]
                 len([None]) == len([...]) → False → 不 append
  
                 → 只在首次需要时 append 一次
            
            '''
            if len(input_tensors[model_chunk_id]) == len(output_tensors[model_chunk_id]):
                '''
                为什么需要 append(None)？
                  PP 首阶段 + VPP 首 chunk 的特殊性：
                    - 输入来自数据加载器，不是 P2P 接收
                    - 但后续代码统一从 input_tensors 取输入
  
                    问题：input_tensors[0] 初始为空列表 []
  
                    解决：append(None) 作为占位符
                    → 后续代码看到 None，知道从 data_iterator 取数据
                
                '''
                input_tensors[model_chunk_id].append(None)

        # For non-depth-first pipeline schedules, the first rank would buffer multiple received
        # activation tensors for a model chunk until accessed during warmup.
        # This input buffering is needed to overlap the computation with the receipt of
        # the next inputs. To index the proper buffered inputs for forword_step, we use
        # microbatch_id offset with number of released microbatches that have completed backprop.
        '''
        输入张量索引（核心难点）
        
    为什么需要偏移量？
        
        非深度优先调度的复杂性：

        深度优先：
          mb0: ch0 → ch1 → 完成
          mb1: ch0 → ch1 → 完成
           → 顺序简单，input_tensors[0] 按序消费
        
        非深度优先（VPP）：
          mb0: ch0
          mb1: ch0
          mb2: ch0
          
          mb0: ch1  ← mb0 的 ch1，但 mb0 的 ch0 输入还在缓存中！
          mb1: ch1
          ...
        
          input_tensors[0] = [mb0_in, mb1_in, mb2_in, mb3_in, mb4_in]
  
          当处理 mb0 的 ch1 时：
            - mb0 的 ch0 已完成前向，但输入还没被消费（因为反向还没开始）
            - 下一个要处理的是 mb0 的 ch1，需要 mb0 的输入
  
          但 input_tensors[0] 里 mb0 的输入在索引 0
          如果直接用 microbatch_id=0，取 input_tensors[0][0] ✓
  
          问题：当处理 mb3 的 ch0 时：
            - mb0, mb1, mb2 的 ch0 已完成前向
            - 但它们的输入可能还没被反向消费（还在缓存中）
            - input_tensors[0] = [mb0_in, mb1_in, mb2_in, mb3_in, mb4_in]（都没释放）
            - microbatch_id=3，取 input_tensors[0][3] ✓
  
          更复杂的情况：某些 mb 已完成反向，输入已释放
            input_tensors[0] = [mb0_in, mb1_in, mb3_in, mb4_in]  （mb2_in 已 pop）
            microbatch_id=3，但 mb3_in 现在在索引 2！
            → 直接用 microbatch_id 会取错！
        '''
        '''
        计算在该 v_id 之前，已有多少个 micro-batch 的输入被释放（完成反向）
        '''
        offset = num_released_microbatches(virtual_microbatch_id, model_chunk_id)
        input_tensor = input_tensors[model_chunk_id][microbatch_id - offset]

        return input_tensor


###  为什么需要判断"首个 micro-batch"？
#### 参数同步场景


In [ ]:
ZeRO-3 参数分片：
  - 每个 GPU 只保存部分参数
  - 前向计算前需要 All-Gather 获取完整参数
  
  chunk 0 的参数：
    - mb0 前向前：需要 All-Gather（首个 mb，参数未同步）
    - mb1 前向前：不需要 All-Gather（参数已在 mb0 时同步）
    - mb2 前向前：不需要 All-Gather
    ...
    
  只在"首个 micro-batch"同步参数，避免重复通信！

is_first_microbatch_for_model_chunk(v_id=0):
  v_id=0 → ch=0, mb=0
  ch=0 的首个 v_id 是 0
  → True！（是首个）

is_first_microbatch_for_model_chunk(v_id=1):
  v_id=1 → ch=0, mb=1
  ch=0 的首个 v_id 是 0，不是 1
  → False！（不是首个）

is_first_microbatch_for_model_chunk(v_id=3):
  v_id=3 → ch=1, mb=0
  ch=1 的首个 v_id 是 3
  → True！（是首个）

#### 梯度同步场景
##### 梯度同步只在"末个 micro-batch"触发（不是首个）


In [ ]:
反向调度表:
v_id | 0  1  2  3  4  5  6  7  8  9
mb   | 0  1  2  0  1  2  3  4  3  4
ch   | 1  1  1  0  0  0  1  1  0  0

chunk 0 的反向（ch=0）:
  v_id=3: mb=0  ← 首个
  v_id=4: mb=1
  v_id=5: mb=2
  v_id=8: mb=3
  v_id=9: mb=4  ← 末个

chunk 1 的反向（ch=1）:
  v_id=0: mb=0  ← 首个
  v_id=1: mb=1
  v_id=2: mb=2
  v_id=6: mb=3
  v_id=7: mb=4  ← 末个

is_last_microbatch_for_model_chunk(v_id=9):
  ch=0 的末个 v_id 是 9
  → True！（是末个，触发 grad_sync）

is_last_microbatch_for_model_chunk(v_id=3):
  ch=0 的末个 v_id 是 9，不是 3
  → False！（不是末个）

## forward_step_helper_postprocess 后处理
- 这段代码是 Megatron-LM 中 VPP（虚拟流水线并行）前向传播后的后处理函数，
- <font color='red'>负责保存输出张量、累加 token 计数和推理模式下的内存清理。</font>
- forward_step_helper_postprocess 是前向传播后的"收尾站"：
  - 保存输出张量到缓存供后续反向传播使用，累加有效 token 数用于全局损失平均，并在推理模式下立即释放输入输出张量——训练时" hoard（囤积）"激活值等待反向，推理时"即用即弃"最大化内存效率。

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│         forward_step_helper_postprocess                      │
│              （前向传播后处理）                               │
├─────────────────────────────────────────────────────────────┤
│  1. 保存输出张量到缓存                                       │
│     output_tensors[chunk].append(output_tensor)              │
├─────────────────────────────────────────────────────────────┤
│  2. 累加有效 token 数                                        │
│     total_num_tokens += num_tokens                            │
├─────────────────────────────────────────────────────────────┤
│  3. 推理模式特殊处理（forward_only=True）                     │
│     - 立即释放输入和输出张量（无需反向传播）                  │
└─────────────────────────────────────────────────────────────┘

In [ ]:

    def forward_step_helper_postprocess(model_chunk_id, output_tensor, num_tokens):
        """Postprocess for forward_step_helper"""
        '''
        保存输出张量；
        
        # 初始化
        output_tensors = [[], []]  # VPP=2，每个 chunk阶段 一个列表

        # 执行后:
         output_tensors[0] = [mb0_out, mb1_out, mb2_out, ...]
         output_tensors[1] = [mb0_out, mb1_out, mb2_out, ...]
         
         为什么需要保存？
         
         反向传播时需要:
         - output_tensor: 重建计算图或检查点
         - output_tensor.grad: 链式法则的起点
  
         如果没有激活检查点:
           output_tensor 直接用于 backward()
    
         如果有激活检查点:
           只需要保存 input_tensor，output_tensor 可以重新计算
           但这里仍然保存，供非检查点层使用
        '''
        output_tensors[model_chunk_id].append(output_tensor)

        nonlocal total_num_tokens
        '''
        累加有效 Token 数
        
        为什么需要 num_tokens？
        
        语言模型训练:
         - 输入有 padding（填充到统一长度）
         - 不是所有位置都参与 loss 计算
  
         num_tokens = 实际非 padding 的 token 数
  
         用途:
           1. Loss 平均: loss = total_loss / total_num_tokens
           2. 吞吐量计算: tokens/sec
           3. 学习率调度: 按实际 token 数调整
        '''
        total_num_tokens += num_tokens

        # If forward-only, no need to save tensors for a backward pass.
        if forward_only:
            '''推理模式-内存清理'''
            # Release the tensor that have completed forward step.
            '''
            pop(0) vs pop()
            
            | 操作                            | 作用        | 原因                              |
            | ----------------------------- | --------- | ------------------------------- |
            | `input_tensors[chunk].pop(0)` | 移除并释放输入张量 | FIFO，移除最早加入的（当前 mb 的输入）         |
            | `output_tensors[chunk].pop()` | 移除并释放输出张量 | LIFO，`pop()` 默认移除最后一个（刚加入的当前输出） |

            '''
            input_tensors[model_chunk_id].pop(0)
            output_tensors[model_chunk_id].pop()

        return


#    forward_step
- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/pipeline_parallel/schedules.py#L316 
- 这段代码是 Megatron-LM 框架中执行模型前向传播（Forward Pass）的核心函数 forward_step。
  - 它负责协调数据加载、模型前向计算、自动混合精度（AMP）、激活值检查点（Activation Checkpointing）以及损失（Loss）计算等一系列关键流程。



## 传入模型的前向传播步骤。

如果是第一阶段，输入张量将从 data_iterator 中获取；
否则，将使用传入的 input_tensor。

### 参数:

    - forward_step_func (callable):
        模型的前向步骤函数，以数据迭代器作为第一个参数，模型作为第二个参数。
        该用户自定义的前向步骤预期需输出一个包含两个元素的元组：

            1. 前向步骤的输出对象。该对象需要是一个张量或某种张量集合。
               对该对象的唯一硬性要求是：它必须能作为第二个函数的输入。
            2. 一个用于（可选地）对前向步骤输出进行归约（reduce）的函数。
               它可以是对模型 loss 的归约，也可以是一个提取并重新格式化模型输出的函数，
               或者仅仅是一个直接透传模型输出的函数。该函数必须符合以下模式之一，
               不同的模式会在内部触发不同的处理逻辑：

                a. 一个包含归约后的 loss 和其他数据的元组。注意，在这种情况下，
                   第一个参数会被除以全局 microbatch 的数量（假设它是一个 loss），
                   从而保证 loss 作为跨设备拆分步数的函数时是稳定的。
                b. 一个包含归约后的 loss、token 数量和其他数据的三元组。
                   这类似于情况 (a)，但 loss 会进一步在 batch 的 token 数量上进行平均。
                   如果用户尚未按 token 数量进行平均，使用此模式会非常有用。
                c. 用户想要的任意数据（例如推理场景下的张量字典、张量列表等）。
                   要触发情况 3，需要指定 `collect_non_loss_data=True`，
                   并且可能还需要在调用父级 forward_backward 函数时指定 `forward_only=True`。

    - data_iterator (iterator):
        数据迭代器。
    - model (nn.Module):
        执行前向步骤的模型。
    - num_microbatches (int):
        microbatch 的数量。
    - input_tensor (Tensor 或 list[Tensor]):
        前向步骤的输入张量。
    - forward_data_store (list):
        用于存储前向数据的列表。如果前向归约函数的返回值走的是 2.a 或 2.b 路径，
        这里将仅存储输出的最后一个维度（例如 loss 函数输出的元数据）。
        如果走的是 2.c 路径，这里将存储对模型输出应用前向归约函数后的完整输出。
    - config (object):
        配置对象。
    - collect_non_loss_data (bool, 可选):
        是否收集非 loss 数据。默认为 False。
        如果你想收集模型前向传播的任意输出（例如推理用例），请使用此路径。默认为 False。
    - checkpoint_activations_microbatch (int, 可选):
        需要进行激活值检查点（checkpoint）的 microbatch。默认为 None。
    - is_first_microbatch (bool, 可选):
        是否为第一个 microbatch。默认为 False。
    - current_microbatch (int, 可选):
        当前的 microbatch。默认为 None。
    - vp_stage (int, 可选):
        虚拟流水线（Virtual Pipeline）阶段。默认为 None。
    - is_last_stage (bool, 可选):
        是否为最后一个阶段。默认为 True。
        此判断同时考虑了虚拟阶段。
        在 PP/VPP 情况下，对应 is_last_stage/is_vp_last_stage。

- 返回:
    - Tensor 或 list[Tensor]: 前向步骤的输出对象。
    - Tensor: token 的数量。
    
## 激活值检查点 (Activation Checkpointing)
- 核心目的：
  - 节省显存（以时间换空间）。在模型训练的前向传播过程中，中间层的激活值（中间结果）会占用大量显存。
    - <font color='red'>激活值检查点通过选择性地丢弃部分中间激活值，在反向传播需要计算梯度时再临时重新计算它们，从而大幅降低显存需求。</font>
- 作用对象：
  - 神经网络前向传播过程中的中间激活张量（Activations）。
- 生命周期：
  - 它是瞬态的，仅存在于单次前向和反向传播的计算过程中，计算完成后即被释放，不会被永久保存到硬盘上。
- 典型应用：
  - 大模型训练时遇到 CUDA Out of Memory（显存溢出）错误，或者希望在有限硬件上训练更大批次（Batch Size）和更长序列的模型。

## 用于恢复的检查点 (Model Checkpointing)
- 核心目的：
  - 容错与状态恢复。在漫长的训练过程中，为了防止因硬件故障、断电或网络中断导致训练进度全部丢失，系统会定期将当前的模型状态保存到磁盘等持久化存储中。
- 作用对象：
  - 模型的全局状态。
    - 这通常包括模型权重（Weights）、优化器状态（如 Adam 的动量和方差）、学习率调度器状态以及当前的训练步数（Epoch/Step）等元数据。
- 生命周期：
  - 它是持久的，会被写入硬盘（如 SSD、NVMe 或分布式文件系统），即使进程重启或机器更换，依然可以加载并继续训练。
- 典型应用：
  - 训练中断后的断点续训（Resume Training）、多轮对话记忆保存（如 LangGraph 中的 Checkpoint）、以及数据库系统中的脏页刷新与故障恢复（如 MySQL、Oracle 中的 Checkpoint 机制）。


In [ ]:

def forward_step(
    forward_step_func,
    data_iterator,
    model,
    num_microbatches,
    input_tensor,
    forward_data_store,
    config,
    cp_group_size,
    collect_non_loss_data=False,
    checkpoint_activations_microbatch=None,
    is_first_microbatch=False,
    current_microbatch=None,
    vp_stage=None,
    is_last_stage=True,
):
    '''
    导入模块：引入了 MTPLossAutoScaler，这通常用于多 Token 预测（Multi-Token Prediction）任务中的损失自动缩放。
    '''
    from megatron.core.transformer.multi_token_prediction import MTPLossAutoScaler

    
    if config.timers is not None:
        config.timers('forward-compute', log_level=2).start()

    '''
    微批次（Microbatch）状态设置
    在梯度累积或多微批次训练中，
    某些模型层（如 MoE 的负载均衡、BatchNorm 等）需要知道当前是否为第一个微批次，
    或者当前是第几个微批次。这段代码负责将这些状态传递给模型内部。
    
    '''
    if is_first_microbatch and hasattr(model, 'set_is_first_microbatch'):
        model.set_is_first_microbatch()
    if current_microbatch is not None:
        set_current_microbatch(model, current_microbatch)

    '''
    输入张量（Input Tensor）标准化与注入:
    
    '''
    unwrap_output_tensor = False
    if not isinstance(input_tensor, list):
        '''
        统一格式：为了兼容流水线并行（Pipeline Parallelism），输入张量被统一转换为列表格式。
        如果原始输入不是列表，
        则将其包装为列表，并记录 unwrap_output_tensor = True，以便在函数返回时恢复原始格式
        '''
        input_tensor = [input_tensor]
        unwrap_output_tensor = True

    '''
    注入输入：通过 set_input_tensor 将输入张量传递给模型。
    在流水线并行的中间阶段，前一阶段的输出就是通过这种方式传递给当前阶段的模型的。
    '''
    set_input_tensor = get_attr_wrapped_model(model, "set_input_tensor")
    set_input_tensor(input_tensor)

    '''
    核心前向计算（支持 AMP 与 激活检查点）:
    
    
    '''
    if config.enable_autocast:
        '''
        自动混合精度（AMP）：如果开启了 autocast，则使用 torch.autocast 上下文管理器，
        使前向计算在指定的低精度（如 FP16/BF16）下运行，以加速计算并节省显存。
        '''
        context_manager = torch.autocast("cuda", dtype=config.autocast_dtype)
    else:
        context_manager = contextlib.nullcontext()
        
    with context_manager:
        '''
        调用用户定义的前向函数：执行 forward_step_func。这个函数通常由用户在训练脚本中定义，
        负责从 data_iterator 获取数据并执行 model.forward()。
        '''
        
        '''
        责任分离与延迟计算：
        前向传播 (forward_step_func) 只负责产出 Logits，
        损失计算被封装为 loss_func 延迟执行。这完美适配了流水线并行中“中间节点无法计算 Loss”的痛点。
        '''
        if checkpoint_activations_microbatch is None:
            
            output_tensor, loss_func = forward_step_func(data_iterator, model)
        else:
            '''
            激活值检查点（Activation Checkpointing）：如果指定了 checkpoint_activations_microbatch，
            则将其作为参数传入。
            这是一种显存优化技术，前向传播时不保存中间激活值，而在反向传播时重新计算，以空间换时间。
            '''
            output_tensor, loss_func = forward_step_func(
                data_iterator, model, checkpoint_activations_microbatch
            )
            '''
            返回值：
            forward_step_func 返回 output_tensor（模型输出/Logits）和 loss_func（一个用于计算损失的函数对象，通常是通过 functools.partial 绑定了标签等参数的闭包）。
            注意：此时并未真正计算损失，而是延迟到下一步。
            '''
     '''
     损失计算与后处理:
     
     这一步将真正的损失计算逻辑剥离到了 forward_step_calc_loss 函数中。
     
     它还负责处理虚拟流水线并行（VPP）、上下文并行（CP）以及非损失数据收集（如推理时的 Logits 收集）等复杂逻辑。
     '''       
    output_tensor, num_tokens = forward_step_calc_loss(
        model,
        output_tensor,
        loss_func,
        config,
        vp_stage,
        collect_non_loss_data,
        num_microbatches,
        forward_data_store,
        cp_group_size,
        is_last_stage, # 它会根据 is_last_stage 判断当前是否为流水线并行的最后一个阶段（只有最后一个阶段才拥有完整的 Logits 和 Labels，才能计算最终的 Loss）。
    )

    if unwrap_output_tensor:
        return output_tensor, num_tokens

    '''
    返回处理后的 output_tensor 和当前批次的 num_tokens（用于后续对 Loss 进行全局平均）
    '''
    return [output_tensor], num_tokens
